In [53]:
%%writefile app.py

import streamlit as st
import json
import os
import re
import jwt
import datetime
import smtplib
import random
from email.message import EmailMessage

JWT_SECRET = "my_super_secret_key_12345"

# Replace these with your Gmail and App Password while developing
EMAIL_ADDRESS = "sahanandani061@gmail.com"
EMAIL_PASSWORD = "dhen ntvy ceoh puqp"

def send_otp(receiver_email):

    otp = str(random.randint(100000, 999999))

    msg = EmailMessage()
    msg["Subject"] = "Your OTP for Password Reset"
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = receiver_email

    msg.set_content(f"Your OTP is: {otp}")

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:
        smtp.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        smtp.send_message(msg)

    return otp


st.set_page_config(
    page_title="User Authentication System",
    page_icon="🔐",
    layout="centered"
)

st.title("🔐 User Authentication System")
# ---------------- DASHBOARD ---------------- #

if "token" in st.session_state:

    st.title("🏠 User Dashboard")

    st.success(f"Welcome, {st.session_state['username']}!")

    if st.button("Logout"):

        del st.session_state["token"]
        del st.session_state["username"]

        st.rerun()

    st.stop()

menu = st.sidebar.radio(
    "Navigation",
    ["Login", "Signup", "Forgot Password", "Admin Login"]
)

# ---------------- LOGIN ---------------- #
# ---------------- LOGIN ---------------- #
if menu == "Login":

    st.header("User Login")

    username = st.text_input("Username / Email")
    password = st.text_input("Password", type="password")

    if st.button("Login"):

        if os.path.exists("users.json"):
            with open("users.json", "r") as f:
                users = json.load(f)
        else:
            users = []

        found = False

        for user in users:

            if (
                (user["username"] == username or user["email"] == username)
                and user["password"] == password
            ):

                found = True

                payload = {
                    "username": user["username"],
                    "email": user["email"],
                    "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=1)
                }

                token = jwt.encode(payload, JWT_SECRET, algorithm="HS256")

                st.session_state["token"] = token
                st.session_state["username"] = user["username"]

                st.success("Login Successful!")

        if not found:
            st.error("Invalid Username or Password")

# ---------------- SIGNUP ---------------- #

# ---------------- SIGNUP ---------------- #

elif menu == "Signup":

    st.header("Create Account")

    username = st.text_input("Username")
    email = st.text_input("Email")
    password = st.text_input("Password", type="password")
    confirm_password = st.text_input("Confirm Password", type="password")

    question = st.selectbox(
        "Security Question",
        [
            "What is your pet's name?",
            "What is your favourite colour?",
            "What is your birthplace?",
            "What is your mother's maiden name?"
        ]
    )

    answer = st.text_input("Security Answer")

    if st.button("Create Account"):

        # 1. Mandatory fields
        if not all([username, email, password, confirm_password, answer]):
            st.error("Please fill all fields.")

        # 2. Confirm Password
        elif password != confirm_password:
            st.error("Passwords do not match.")

        # 3. Email Validation
        elif not re.match(r'^[A-Za-z0-9._%+-]{2,}@[A-Za-z]{2,}\.[A-Za-z]{2,}$', email):
            st.error("Enter a valid email address.")

        # 4. Password Validation
        elif not re.match(r'^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[@$!%*?&]).{8,}$', password):
            st.error(
                "Password must be at least 8 characters and contain uppercase, lowercase, number and special symbol."
            )

        else:

            # Load existing users
            if os.path.exists("users.json"):
                with open("users.json", "r") as f:
                    users = json.load(f)
            else:
                users = []

            # Check duplicate username
            for user in users:
                if user["username"] == username:
                    st.error("Username already exists.")
                    st.stop()

            # Save new user
            users.append({
                "username": username,
                "email": email,
                "password": password,
                "security_question": question,
                "security_answer": answer
            })

            with open("users.json", "w") as f:
                json.dump(users, f, indent=4)

            st.success("Account created successfully!")


# ---------------- FORGOT PASSWORD ---------------- #

elif menu == "Forgot Password":

    st.header("Forgot Password")

    option = st.radio(
        "Choose Recovery Method",
        [
            "Security Question",
            "OTP via Email"
        ]
    )

    # ---------- Security Question Recovery ---------- #

    if option == "Security Question":

        username = st.text_input("Username")

        if username:

            if os.path.exists("users.json"):
                with open("users.json", "r") as f:
                    users = json.load(f)
            else:
                users = []

            current_user = None

            for user in users:
                if user["username"] == username:
                    current_user = user
                    break

            if current_user:

                st.write("### Security Question")
                st.info(current_user["security_question"])

                answer = st.text_input("Security Answer")

                new_password = st.text_input("New Password", type="password")

                confirm_password = st.text_input("Confirm Password", type="password")

                if st.button("Reset Password"):

                    if answer != current_user["security_answer"]:
                        st.error("Incorrect security answer.")

                    elif new_password != confirm_password:
                        st.error("Passwords do not match.")

                    elif not re.match(
                        r'^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[@$!%*?&]).{8,}$',
                        new_password
                    ):
                        st.error(
                            "Password must contain uppercase, lowercase, number and special symbol."
                        )

                    else:

                        current_user["password"] = new_password

                        with open("users.json", "w") as f:
                            json.dump(users, f, indent=4)

                        st.success("Password updated successfully!")

            else:
                st.error("Username not found.")

    # ---------- OTP Recovery ---------- #

    else:

        email = st.text_input("Registered Email")

        if st.button("Send OTP"):

            if os.path.exists("users.json"):
                with open("users.json", "r") as f:
                    users = json.load(f)
            else:
                users = []

            registered = False

            for user in users:

                if user["email"] == email:

                    registered = True

                    otp = send_otp(email)

                    st.session_state["otp"] = otp
                    st.session_state["reset_email"] = email

                    st.success("OTP sent successfully!")

                    break

            if not registered:
                st.error("Email not registered.")

        entered_otp = st.text_input("Enter OTP")

        new_password = st.text_input("New Password", type="password")

        confirm_password = st.text_input("Confirm Password", type="password")

        if st.button("Verify OTP"):

            if "otp" not in st.session_state:
                st.error("Please generate an OTP first.")

            elif entered_otp != st.session_state["otp"]:
                st.error("Invalid OTP.")

            elif new_password != confirm_password:
                st.error("Passwords do not match.")

            elif not re.match(
                r'^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[@$!%*?&]).{8,}$',
                new_password
            ):
                st.error(
                    "Password must contain uppercase, lowercase, number and special symbol."
                )

            else:

                with open("users.json", "r") as f:
                    users = json.load(f)

                for user in users:

                    if user["email"] == st.session_state["reset_email"]:

                        user["password"] = new_password
                        break

                with open("users.json", "w") as f:
                    json.dump(users, f, indent=4)

                del st.session_state["otp"]
                del st.session_state["reset_email"]

                st.success("Password updated successfully!")

# ---------------- ADMIN ---------------- #

elif menu == "Admin Login":

    st.header("Admin Login")

    admin = st.text_input("Admin Username")

    password = st.text_input("Admin Password", type="password")

    if st.button("Admin Login"):
        st.info("Admin Dashboard will be added later.")

Overwriting app.py


In [54]:
import smtplib

EMAIL = "sahanandani061@gmail.com"
PASSWORD = "dhen ntvy ceoh puqp"

try:
    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:
        smtp.login(EMAIL, PASSWORD)
    print("✅ Gmail Login Successful!")
except Exception as e:
    print("❌", e)

✅ Gmail Login Successful!


In [ ]:
%%writefile app.py

import streamlit as st
import json
import os
import re
import jwt
import datetime
import smtplib
import random
from email.message import EmailMessage

JWT_SECRET = "my_super_secret_key_12345"

# Replace these with your Gmail and App Password while developing
EMAIL_ADDRESS = "sahanandani061@gmail.com"
EMAIL_PASSWORD = "dhen ntvy ceoh puqp"

def send_otp(receiver_email):

    otp = str(random.randint(100000, 999999))

    msg = EmailMessage()
    msg["Subject"] = "Your OTP for Password Reset"
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = receiver_email

    msg.set_content(f"Your OTP is: {otp}")

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:
        smtp.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        smtp.send_message(msg)

    return otp


st.set_page_config(
    page_title="User Authentication System",
    page_icon="🔐",
    layout="centered"
)

st.title("🔐 User Authentication System")
# ---------------- DASHBOARD ---------------- #

if "token" in st.session_state:

    st.title("🏠 User Dashboard")

    st.success(f"Welcome, {st.session_state['username']}!")

    if st.button("Logout"):

        del st.session_state["token"]
        del st.session_state["username"]

        st.rerun()

    st.stop()

menu = st.sidebar.radio(
    "Navigation",
    ["Login", "Signup", "Forgot Password", "Admin Login"]
)

# ---------------- LOGIN ---------------- #
# ---------------- LOGIN ---------------- #
if menu == "Login":

    st.header("User Login")

    username = st.text_input("Username / Email")
    password = st.text_input("Password", type="password")

    if st.button("Login"):

        if os.path.exists("users.json"):
            with open("users.json", "r") as f:
                users = json.load(f)
        else:
            users = []

        found = False

        for user in users:

            if (
                (user["username"] == username or user["email"] == username)
                and user["password"] == password
            ):

                found = True

                payload = {
                    "username": user["username"],
                    "email": user["email"],
                    "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=1)
                }

                token = jwt.encode(payload, JWT_SECRET, algorithm="HS256")

                st.session_state["token"] = token
                st.session_state["username"] = user["username"]

                st.success("Login Successful!")

        if not found:
            st.error("Invalid Username or Password")

# ---------------- SIGNUP ---------------- #

# ---------------- SIGNUP ---------------- #

elif menu == "Signup":

    st.header("Create Account")

    username = st.text_input("Username")
    email = st.text_input("Email")
    password = st.text_input("Password", type="password")
    confirm_password = st.text_input("Confirm Password", type="password")

    question = st.selectbox(
        "Security Question",
        [
            "What is your pet's name?",
            "What is your favourite colour?",
            "What is your birthplace?",
            "What is your mother's maiden name?"
        ]
    )

    answer = st.text_input("Security Answer")

    if st.button("Create Account"):

        # 1. Mandatory fields
        if not all([username, email, password, confirm_password, answer]):
            st.error("Please fill all fields.")

        # 2. Confirm Password
        elif password != confirm_password:
            st.error("Passwords do not match.")

        # 3. Email Validation
        elif not re.match(r'^[A-Za-z0-9._%+-]{2,}@[A-Za-z]{2,}\.[A-Za-z]{2,}$', email):
            st.error("Enter a valid email address.")

        # 4. Password Validation
        elif not re.match(r'^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[@$!%*?&]).{8,}$', password):
            st.error(
                "Password must be at least 8 characters and contain uppercase, lowercase, number and special symbol."
            )

        else:

            # Load existing users
            if os.path.exists("users.json"):
                with open("users.json", "r") as f:
                    users = json.load(f)
            else:
                users = []

            # Check duplicate username
            for user in users:
                if user["username"] == username:
                    st.error("Username already exists.")
                    st.stop()

            # Save new user
            users.append({
                "username": username,
                "email": email,
                "password": password,
                "security_question": question,
                "security_answer": answer
            })

            with open("users.json", "w") as f:
                json.dump(users, f, indent=4)

            st.success("Account created successfully!")


# ---------------- FORGOT PASSWORD ---------------- #

elif menu == "Forgot Password":

    st.header("Forgot Password")

    option = st.radio(
        "Choose Recovery Method",
        [
            "Security Question",
            "OTP via Email"
        ]
    )

    # ---------- Security Question Recovery ---------- #

    if option == "Security Question":

        username = st.text_input("Username")

        if username:

            if os.path.exists("users.json"):
                with open("users.json", "r") as f:
                    users = json.load(f)
            else:
                users = []

            current_user = None

            for user in users:
                if user["username"] == username:
                    current_user = user
                    break

            if current_user:

                st.write("### Security Question")
                st.info(current_user["security_question"])

                answer = st.text_input("Security Answer")

                new_password = st.text_input("New Password", type="password")

                confirm_password = st.text_input("Confirm Password", type="password")

                if st.button("Reset Password"):

                    if answer != current_user["security_answer"]:
                        st.error("Incorrect security answer.")

                    elif new_password != confirm_password:
                        st.error("Passwords do not match.")

                    elif not re.match(
                        r'^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[@$!%*?&]).{8,}$',
                        new_password
                    ):
                        st.error(
                            "Password must contain uppercase, lowercase, number and special symbol."
                        )

                    else:

                        current_user["password"] = new_password

                        with open("users.json", "w") as f:
                            json.dump(users, f, indent=4)

                        st.success("Password updated successfully!")

            else:
                st.error("Username not found.")

    # ---------- OTP Recovery ---------- #

    else:

        email = st.text_input("Registered Email")

        if st.button("Send OTP"):

            if os.path.exists("users.json"):
                with open("users.json", "r") as f:
                    users = json.load(f)
            else:
                users = []

            registered = False

            for user in users:

                if user["email"] == email:

                    registered = True

                    otp = send_otp(email)

                    st.session_state["otp"] = otp
                    st.session_state["reset_email"] = email

                    st.success("OTP sent successfully!")

                    break

            if not registered:
                st.error("Email not registered.")

        entered_otp = st.text_input("Enter OTP")

        new_password = st.text_input("New Password", type="password")

        confirm_password = st.text_input("Confirm Password", type="password")

        if st.button("Verify OTP"):

            if "otp" not in st.session_state:
                st.error("Please generate an OTP first.")

            elif entered_otp != st.session_state["otp"]:
                st.error("Invalid OTP.")

            elif new_password != confirm_password:
                st.error("Passwords do not match.")

            elif not re.match(
                r'^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[@$!%*?&]).{8,}$',
                new_password
            ):
                st.error(
                    "Password must contain uppercase, lowercase, number and special symbol."
                )

            else:

                with open("users.json", "r") as f:
                    users = json.load(f)

                for user in users:

                    if user["email"] == st.session_state["reset_email"]:

                        user["password"] = new_password
                        break

                with open("users.json", "w") as f:
                    json.dump(users, f, indent=4)

                del st.session_state["otp"]
                del st.session_state["reset_email"]

                st.success("Password updated successfully!")

# ---------------- ADMIN ---------------- #

elif menu == "Admin Login":

    st.header("Admin Login")

    admin = st.text_input("Admin Username")

    password = st.text_input("Admin Password", type="password")

    if st.button("Admin Login"):
        st.info("Admin Dashboard will be added later.")

Overwriting app.py


In [1]:
print("Milestone 1 Started Successfully!")

Milestone 1 Started Successfully!


In [2]:
!pip install streamlit pyngrok pyjwt bcrypt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 70.8 MB/s eta 0:00:00


In [4]:
!cat app.py


import streamlit as st

st.set_page_config(
    page_title="Milestone 1",
    page_icon="🔐",
    layout="centered"
)

st.title("🔐 User Authentication System")
st.subheader("Infosys Springboard Internship 7.0")
st.write("Welcome to Milestone 1!")

st.success("🎉 Streamlit is working successfully!")


In [6]:
!streamlit run app.py &>/content/logs.txt &

In [7]:
!cat /content/logs.txt



2026-07-07 19:04:47.982 Uvicorn server started on :::8502

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://34.86.165.36:8502



In [39]:
from pyngrok import ngrok
from google.colab import userdata

ngrok.kill()

ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))

public_url = ngrok.connect(8501)

print("Open this URL:")
print(public_url)

Open this URL:
NgrokTunnel: "https://bootlace-semicolon-reliable.ngrok-free.dev" -> "http://localhost:8501"


In [15]:
!pkill -f streamlit

In [16]:
!streamlit run app.py &



2026-07-07 19:22:05.731 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.86.165.36:8501



  Stopping...


In [18]:
!nohup streamlit run app.py --server.port 8501 > streamlit.log 2>&1 &

In [19]:
!cat streamlit.log



2026-07-07 19:30:41.632 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.86.165.36:8501



In [20]:
from pyngrok import ngrok
from google.colab import userdata

ngrok.kill()

ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))

public_url = ngrok.connect(8501)

print("Open this URL:")
print(public_url)

Open this URL:
NgrokTunnel: "https://bootlace-semicolon-reliable.ngrok-free.dev" -> "http://localhost:8501"


In [23]:
!pkill -f streamlit

In [24]:
!nohup streamlit run app.py --server.port 8501 > streamlit.log 2>&1 &

In [25]:
import json
import os

if not os.path.exists("users.json"):
    with open("users.json", "w") as f:
        json.dump([], f)

print("users.json created successfully!")

users.json created successfully!


In [26]:
!cat users.json

[]

In [29]:
!nohup streamlit run app.py --server.port 8501 > streamlit.log 2>&1 &

In [30]:
!cat streamlit.log



2026-07-07 20:18:04.859 Port 8501 is not available


In [31]:
!ps -ef | grep streamlit

root       13004       1  0 19:50 ?        00:00:08 /usr/bin/python3 /usr/local/bin/streamlit run app.py --server.port 8501
root       20283     661  0 20:20 ?        00:00:00 /bin/bash -c ps -ef | grep streamlit
root       20285   20283  0 20:20 ?        00:00:00 grep streamlit


In [35]:
!pkill -f streamlit

In [36]:
!nohup streamlit run app.py --server.port 8501 > streamlit.log 2>&1 &!cat streamlit.log

In [37]:
!cat streamlit.log



2026-07-07 20:35:52.400 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.86.165.36:8501



In [40]:
!cat users.json

[
    {
        "username": "rishi",
        "email": "rishi123@gmail.com",
        "password": "zxcvbnm",
        "security_question": "What is your pet's name?",
        "security_answer": "bobo"
    }
]

In [41]:
!pip install PyJWT

In [5]:
!cat streamlit.log

cat: streamlit.log: No such file or directory


In [6]:
!streamlit --version

/bin/bash: line 1: streamlit: command not found


In [7]:
!pip install streamlit pyngrok pyjwt bcrypt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 36.8 MB/s eta 0:00:00


In [29]:
!pkill -f streamlit

In [30]:
!nohup streamlit run app.py --server.port 8501 > streamlit.log 2>&1 &

In [31]:
!cat streamlit.log



2026-07-08 03:05:08.288 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.234.183.254:8501



In [18]:
!pkill -f streamlit

In [20]:
!nohup streamlit run app.py --server.port 8501 > streamlit.log 2>&1 &

In [9]:
!ls -l

total 12
-rw-r--r-- 1 root root 4473 Jul  8 02:23 app.py
drwxr-xr-x 1 root root 4096 Jun  4 13:32 sample_data


In [10]:
!python app.py

2026-07-08 02:24:40.613 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 02:24:40.614 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 02:24:40.680 
  command:

    streamlit run app.py [ARGUMENTS]
2026-07-08 02:24:40.680 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 02:24:40.680 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 02:24:40.680 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 02:24:40.681 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 02:24:40.681 Thread 'MainThre

In [11]:
!nohup streamlit run app.py --server.port 8501 > streamlit.log 2>&1 &

In [12]:
!cat streamlit.log



2026-07-08 02:25:37.479 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.234.183.254:8501



In [13]:
from pyngrok import ngrok
from google.colab import userdata

ngrok.kill()

ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))

public_url = ngrok.connect(8501)

print("Open this URL:")
print(public_url)

Open this URL:
NgrokTunnel: "https://bootlace-semicolon-reliable.ngrok-free.dev" -> "http://localhost:8501"


In [15]:
!pip install pyjwt

In [21]:
!ls -l users.json

-rw-r--r-- 1 root root 210 Jul  8 02:34 users.json


In [22]:
!cat users.json

[
    {
        "username": "rishi",
        "email": "rishi123@gmail.com",
        "password": "Qwertyuiop@1",
        "security_question": "What is your pet's name?",
        "security_answer": "bobo"
    }
]

In [23]:
!pip install bcrypt

In [33]:
!pkill -f streamlit

In [34]:
!nohup streamlit run app.py --server.port 8501 > streamlit.log 2>&1 &

In [52]:
%%writefile app.py

import streamlit as st
import json
import os
import re
import jwt
import datetime
import smtplib
import random
from email.message import EmailMessage

JWT_SECRET = "my_super_secret_key_12345"

# Replace these with your Gmail and App Password while developing
EMAIL_ADDRESS = "sahanandani061@gmail.com"
EMAIL_PASSWORD = "dhen ntvy ceoh puqp"

def send_otp(receiver_email):

    otp = str(random.randint(100000, 999999))

    msg = EmailMessage()
    msg["Subject"] = "Your OTP for Password Reset"
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = receiver_email

    msg.set_content(f"Your OTP is: {otp}")

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:
        smtp.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        smtp.send_message(msg)

    return otp


st.set_page_config(
    page_title="User Authentication System",
    page_icon="🔐",
    layout="centered"
)

st.title("🔐 User Authentication System")
# ---------------- DASHBOARD ---------------- #

if "token" in st.session_state:

    st.title("🏠 User Dashboard")

    st.success(f"Welcome, {st.session_state['username']}!")

    if st.button("Logout"):

        del st.session_state["token"]
        del st.session_state["username"]

        st.rerun()

    st.stop()

menu = st.sidebar.radio(
    "Navigation",
    ["Login", "Signup", "Forgot Password", "Admin Login"]
)

# ---------------- LOGIN ---------------- #
# ---------------- LOGIN ---------------- #
if menu == "Login":

    st.header("User Login")

    username = st.text_input("Username / Email")
    password = st.text_input("Password", type="password")

    if st.button("Login"):

        if os.path.exists("users.json"):
            with open("users.json", "r") as f:
                users = json.load(f)
        else:
            users = []

        found = False

        for user in users:

            if (
                (user["username"] == username or user["email"] == username)
                and user["password"] == password
            ):

                found = True

                payload = {
                    "username": user["username"],
                    "email": user["email"],
                    "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=1)
                }

                token = jwt.encode(payload, JWT_SECRET, algorithm="HS256")

                st.session_state["token"] = token
                st.session_state["username"] = user["username"]

                st.success("Login Successful!")

        if not found:
            st.error("Invalid Username or Password")

# ---------------- SIGNUP ---------------- #

# ---------------- SIGNUP ---------------- #

elif menu == "Signup":

    st.header("Create Account")

    username = st.text_input("Username")
    email = st.text_input("Email")
    password = st.text_input("Password", type="password")
    confirm_password = st.text_input("Confirm Password", type="password")

    question = st.selectbox(
        "Security Question",
        [
            "What is your pet's name?",
            "What is your favourite colour?",
            "What is your birthplace?",
            "What is your mother's maiden name?"
        ]
    )

    answer = st.text_input("Security Answer")

    if st.button("Create Account"):

        # 1. Mandatory fields
        if not all([username, email, password, confirm_password, answer]):
            st.error("Please fill all fields.")

        # 2. Confirm Password
        elif password != confirm_password:
            st.error("Passwords do not match.")

        # 3. Email Validation
        elif not re.match(r'^[A-Za-z0-9._%+-]{2,}@[A-Za-z]{2,}\.[A-Za-z]{2,}$', email):
            st.error("Enter a valid email address.")

        # 4. Password Validation
        elif not re.match(r'^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[@$!%*?&]).{8,}$', password):
            st.error(
                "Password must be at least 8 characters and contain uppercase, lowercase, number and special symbol."
            )

        else:

            # Load existing users
            if os.path.exists("users.json"):
                with open("users.json", "r") as f:
                    users = json.load(f)
            else:
                users = []

            # Check duplicate username
            for user in users:
                if user["username"] == username:
                    st.error("Username already exists.")
                    st.stop()

            # Save new user
            users.append({
                "username": username,
                "email": email,
                "password": password,
                "security_question": question,
                "security_answer": answer
            })

            with open("users.json", "w") as f:
                json.dump(users, f, indent=4)

            st.success("Account created successfully!")


# ---------------- FORGOT PASSWORD ---------------- #

elif menu == "Forgot Password":

    st.header("Forgot Password")

    option = st.radio(
        "Choose Recovery Method",
        [
            "Security Question",
            "OTP via Email"
        ]
    )

    # ---------- Security Question Recovery ---------- #

    if option == "Security Question":

        username = st.text_input("Username")

        if username:

            if os.path.exists("users.json"):
                with open("users.json", "r") as f:
                    users = json.load(f)
            else:
                users = []

            current_user = None

            for user in users:
                if user["username"] == username:
                    current_user = user
                    break

            if current_user:

                st.write("### Security Question")
                st.info(current_user["security_question"])

                answer = st.text_input("Security Answer")

                new_password = st.text_input("New Password", type="password")

                confirm_password = st.text_input("Confirm Password", type="password")

                if st.button("Reset Password"):

                    if answer != current_user["security_answer"]:
                        st.error("Incorrect security answer.")

                    elif new_password != confirm_password:
                        st.error("Passwords do not match.")

                    elif not re.match(
                        r'^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[@$!%*?&]).{8,}$',
                        new_password
                    ):
                        st.error(
                            "Password must contain uppercase, lowercase, number and special symbol."
                        )

                    else:

                        current_user["password"] = new_password

                        with open("users.json", "w") as f:
                            json.dump(users, f, indent=4)

                        st.success("Password updated successfully!")

            else:
                st.error("Username not found.")

    # ---------- OTP Recovery ---------- #

    else:

        email = st.text_input("Registered Email")

        if st.button("Send OTP"):

            if os.path.exists("users.json"):
                with open("users.json", "r") as f:
                    users = json.load(f)
            else:
                users = []

            registered = False

            for user in users:

                if user["email"] == email:

                    registered = True

                    otp = send_otp(email)

                    st.session_state["otp"] = otp
                    st.session_state["reset_email"] = email

                    st.success("OTP sent successfully!")

                    break

            if not registered:
                st.error("Email not registered.")

        entered_otp = st.text_input("Enter OTP")

        new_password = st.text_input("New Password", type="password")

        confirm_password = st.text_input("Confirm Password", type="password")

        if st.button("Verify OTP"):

            if "otp" not in st.session_state:
                st.error("Please generate an OTP first.")

            elif entered_otp != st.session_state["otp"]:
                st.error("Invalid OTP.")

            elif new_password != confirm_password:
                st.error("Passwords do not match.")

            elif not re.match(
                r'^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[@$!%*?&]).{8,}$',
                new_password
            ):
                st.error(
                    "Password must contain uppercase, lowercase, number and special symbol."
                )

            else:

                with open("users.json", "r") as f:
                    users = json.load(f)

                for user in users:

                    if user["email"] == st.session_state["reset_email"]:

                        user["password"] = new_password
                        break

                with open("users.json", "w") as f:
                    json.dump(users, f, indent=4)

                del st.session_state["otp"]
                del st.session_state["reset_email"]

                st.success("Password updated successfully!")

# ---------------- ADMIN ---------------- #

elif menu == "Admin Login":

    st.header("Admin Login")

    admin = st.text_input("Admin Username")

    password = st.text_input("Admin Password", type="password")

    if st.button("Admin Login"):
        st.info("Admin Dashboard will be added later.")

Overwriting app.py


In [50]:
!pkill -f streamlit

In [51]:
!nohup streamlit run app.py --server.port 8501 > streamlit.log 2>&1 &

In [45]:
!pkill -f streamlit

In [46]:
!nohup streamlit run app.py --server.port 8501 > streamlit.log 2>&1 &

In [47]:
!cat streamlit.log



2026-07-08 04:40:35.302 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.234.183.254:8501



In [37]:
!pip install secure-smtplib